# Chicago Crime Analysis

Ce notebook regroupe les quatre requêtes du projet d'analyse de la criminalité à Chicago :

| Requête | Thème | Auteure |
|---|---|---|
| Requête 1 | Data Exploration | Angelikia Kavuansiko |
| Requête 2 | Pattern Mining & Apriori | Ekta |
| Requête 3 | Analyse temporelle & Forecasting | Léora |
| Requête 4 | Analyse spatiale & Clustering | Chrisa |

Chaque méthode est encapsulée dans une fonction documentée (nom, entrées, sorties), et chaque
visualisation suit immédiatement son code et son explication. Un bloc principal consolidé
appelle toutes les fonctions et lance le dashboard Dash en fin de notebook.

> **Sources de données :** les Requêtes 1, 2 et 4 chargent les données depuis l'API du Chicago
> Data Portal. La **Requête 3** charge les données depuis le fichier CSV `../data/chicago_crime.csv`, filtrant sur un seul quartier : Bridgeton.

**Lien du dépôt GitHub :** https://github.com/akavuansiko/chicago-crime-analysis

---

## 1. Objectif et démarche KDD

La problématique du projet est la suivante :

> **Comment les crimes à Chicago se distribuent-ils dans l'espace et dans le temps, et peut-on identifier des patterns récurrents pour mieux anticiper leur évolution ?**

Le notebook suit les principales étapes du processus **Knowledge Discovery in Databases (KDD)** :

1. compréhension et chargement des données ;
2. nettoyage et transformation ;
3. construction d'indicateurs descriptifs ;
4. recherche de motifs fréquents ;
5. analyse temporelle et prévision ;
6. analyse spatiale et clustering ;
7. intégration des résultats dans un dashboard.

Toutes les méthodes importantes sont regroupées dans des fonctions documentées avec leur **nom**, leurs **entrées** et leurs **sorties**.

## 2. Déclaration relative à l'utilisation d'un LLM

Une aide par modèle de langage a été utilisée pour consolider, commenter et sécuriser le code provenant des quatre notebooks de travail.

**Prompt utilisé :**

> « À partir des notebooks `exploration.ipynb`, `pattern_mining.ipynb`, `temporal_analysis.ipynb` et `spatial_analysis.ipynb`, créer un notebook final consolidé. Organiser chaque méthode dans une fonction documentée avec son nom, ses entrées et ses sorties. Placer chaque visualisation immédiatement après son code et son explication. Ajouter un bloc principal appelant toutes les fonctions et prévoir le lancement du dashboard Dash. »


**Prompt utilisé pour le remplacement OPTICS → DBSCAN :**

> « L'algorithme OPTICS appliqué sur 10 000 points GPS couvrant toute la ville de Chicago
> mettait tous les crimes dans un seul cluster — impossible d'obtenir des zones distinctes.
> Remplace OPTICS par DBSCAN avec metric="haversine" et algorithm="ball_tree",
> en convertissant les coordonnées en radians et en fixant eps=0.5/6371
> (rayon de 0.5 km converti en radians, 6371 = rayon de la Terre en km).
> Affiche les crimes isolés (-1) en gris transparent en arrière-plan
> et les hotspots colorés par-dessus pour une meilleure lisibilité visuelle. »

Le code reste relu, testé et interprété par l'équipe. L'utilisation du LLM ne remplace pas la compréhension des méthodes employées.

## 3. Importation des bibliothèques et configuration

In [2]:
from pathlib import Path
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import KMeans, OPTICS
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from IPython.display import display

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except Exception as exc:
    Prophet = None
    PROPHET_AVAILABLE = False
    print(f"Prophet indisponible : {exc}")

warnings.filterwarnings("ignore")
pio.renderers.default = "iframe_connected"
pd.set_option("display.max_columns", 50)
print("Imports OK")

Imports OK


## 4. Chargement des données

Le dataset Chicago Crime provient du portail open data de la ville de Chicago ([City of Chicago Data Portal](https://data.cityofchicago.org)).
Il contient les incidents criminels déclarés à Chicago, extraits du système CLEAR (Chicago Police Department).

On charge **10 000 crimes récents** directement depuis l'API officielle pour couvrir l'ensemble de la ville.
La colonne `Date` est parsée en datetime pour permettre les analyses temporelles.
Un fix SSL est appliqué automatiquement sur Mac (Python 3.12).

In [3]:
# Fonction : resolve_project_root
# Input  : aucun
# Output : chemin Path correspondant à la racine du projet
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data").exists() or (candidate / "notebooks").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


# Fonction : load_data
# Input  : limit = nombre de crimes à récupérer depuis l'API (défaut 10 000)
# Output : DataFrame pandas avec dates et coordonnées converties
def load_data(limit: int = 10000) -> pd.DataFrame:
    """Charger les données depuis l'API officielle Chicago Data Portal."""
    url = (
        f"https://data.cityofchicago.org/resource/ijzp-q8t2.csv"
        f"?$limit={limit}"
        f"&$order=date%20DESC"
        f"&$where=latitude%20IS%20NOT%20NULL"
    )

    # Fix SSL uniquement sur Mac (Python 3.12 ne trouve pas les certificats système)
    if sys.platform == "darwin":
        import requests, io, certifi
        response = requests.get(url, verify=certifi.where())
        df = pd.read_csv(io.StringIO(response.text))
    else:
        df = pd.read_csv(url)

    # Renommage des colonnes (l'API retourne en minuscules)
    df = df.rename(columns={
        'latitude'            : 'Latitude',
        'longitude'           : 'Longitude',
        'primary_type'        : 'Primary Type',
        'arrest'              : 'Arrest',
        'date'                : 'Date',
        'location_description': 'Location Description',
        'domestic'            : 'Domestic',
        'beat'                : 'Beat',
        'ward'                : 'Ward',
        'fbi_code'            : 'FBI Code',
        'year'                : 'Year',
        'description'         : 'Description',
    })

    df['Date']      = pd.to_datetime(df['Date'], errors='coerce')
    df['Latitude']  = pd.to_numeric(df['Latitude'],  errors='coerce')
    df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
    df['Year']      = df['Date'].dt.year.astype('Int64')
    df['Month']     = df['Date'].dt.month.astype('Int64')
    df['Hour']      = df['Date'].dt.hour.astype('Int64')
    df['YearMonth'] = df['Date'].dt.to_period('M').dt.to_timestamp()

    print(f"Dataset chargé : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    return df


df = load_data()
df.head()

Dataset chargé : 10000 lignes x 25 colonnes


,id,case_number,Date,block,iucr,Primary Type,Description,Location Description,Arrest,Domestic,Beat,district,Ward,community_area,FBI Code,x_coordinate,y_coordinate,Year,updated_on,Latitude,Longitude,location,Month,Hour,YearMonth
0,14223030,JK286094,2026-06-08,036XX W DOUGLAS BLVD,0497,BATTERY,AGGRAVATED DOMESTIC BATTERY - OTHER DANGEROUS ...,APARTMENT,False,True,1011,10,24,29,04B,1152476,1893218,2026,2026-06-15T15:46:40.000,41.862851,-87.715755,"\n, \n(41.862850968, -87.715755)",6,0,2026-06-01
1,14223973,JK287246,2026-06-08,072XX S RICHMOND ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,831,8,18,66,08B,1157959,1856471,2026,2026-06-15T15:46:40.000,41.761902,-87.696627,"\n, \n(41.76190249, -87.696626851)",6,0,2026-06-01
2,14223471,JK286728,2026-06-08,076XX S MORGAN ST,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,612,6,17,71,07,1170965,1854265,2026,2026-06-15T15:46:40.000,41.755575,-87.649023,"\n, \n(41.755574885, -87.649022623)",6,0,2026-06-01
3,14224250,JK286700,2026-06-08,061XX N KIRKWOOD AVE,0930,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,STREET,False,False,1711,17,39,12,07,1145138,1940565,2026,2026-06-15T15:46:40.000,41.992917,-87.741493,"\n, \n(41.992917396, -87.741492591)",6,0,2026-06-01
4,14225502,JK287707,2026-06-08,040XX W DICKENS AVE,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,RESIDENCE,False,False,2525,25,35,20,26,1148906,1913645,2026,2026-06-15T15:46:40.000,41.918975,-87.728331,"\n, \n(41.918974575, -87.728331482)",6,0,2026-06-01


## 5. Requête 1 — Data Exploration
**Auteure : Angelikia Kavuansiko**

**Problématique :** Comment les crimes à Chicago se distribuent-ils dans l'espace et dans le temps, et peut-on identifier des patterns récurrents pour mieux anticiper leur évolution ?

---

##  Exploration complète du dataset

### Description des colonnes principales

| Colonne | Description |
|---|---|
| `Date` | Date et heure de l'incident |
| `Primary Type` | Type de crime (vol, agression, meurtre...) |
| `Description` | Description détaillée du crime |
| `Location Description` | Lieu de l'incident (rue, parking, école...) |
| `Arrest` | Arrestation effectuée (True/False) |
| `Beat` | Zone de patrouille de police |
| `Ward` | Circonscription électorale |
| `FBI Code` | Classification FBI du crime |
| `Latitude` | Latitude géographique |
| `Longitude` | Longitude géographique |

In [4]:
# Fonction : explore_shape
# Input  : df = DataFrame des crimes
# Output : affiche les dimensions du dataset (aucun retour)
def explore_shape(df):
    # Dimensions du dataset
    print(f" Shape : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

explore_shape(df)

 Shape : 10,000 lignes × 25 colonnes


In [5]:
# Fonction : explore_dtypes
# Input  : df = DataFrame des crimes
# Output : affiche les types de chaque colonne (aucun retour)
def explore_dtypes(df):
    # Types de colonnes
    print(" Types de données :")
    print(df.dtypes)

explore_dtypes(df)

 Types de données :
id                               int64
case_number                     object
Date                    datetime64[ns]
block                           object
iucr                            object
Primary Type                    object
Description                     object
Location Description            object
Arrest                            bool
Domestic                          bool
Beat                             int64
district                         int64
Ward                             int64
community_area                   int64
FBI Code                        object
x_coordinate                     int64
y_coordinate                     int64
Year                             Int64
updated_on                      object
Latitude                       float64
Longitude                      float64
location                        object
Month                            Int64
Hour                             Int64
YearMonth               datetime64[ns]
dtype

In [6]:
# Fonction : explore_missing
# Input  : df = DataFrame des crimes
# Output : DataFrame des colonnes contenant des valeurs manquantes
def explore_missing(df):
    # Valeurs manquantes
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"Manquantes": missing, "%": missing_pct})
    print(" Valeurs manquantes :")
    return missing_df[missing_df["Manquantes"] > 0]

explore_missing(df)

 Valeurs manquantes :


,Manquantes,%
Location Description,20,0.2


In [7]:
# Fonction : explore_ranges
# Input  : df = DataFrame des crimes
# Output : affiche la période couverte, les wards et le nombre de types de crimes
def explore_ranges(df):
    # Ranges
    print(f" Période couverte : {df['Date'].min()} → {df['Date'].max()}")
    print(f" Wards : {df['Ward'].dropna().astype(int).unique().tolist()}")
    print(f" Types de crimes uniques : {df['Primary Type'].nunique()}")

explore_ranges(df)

 Période couverte : 2026-05-23 20:30:00 → 2026-06-08 00:00:00
 Wards : [24, 18, 17, 39, 35, 7, 31, 42, 44, 4, 25, 10, 16, 34, 2, 9, 3, 19, 5, 48, 13, 8, 26, 45, 41, 37, 22, 1, 27, 6, 20, 30, 28, 15, 47, 36, 46, 14, 33, 29, 23, 21, 12, 11, 43, 40, 50, 49, 38, 32]
 Types de crimes uniques : 28


In [8]:
# Fonction : explore_describe
# Input  : df = DataFrame des crimes
# Output : statistiques descriptives (DataFrame)
def explore_describe(df):
    # Stats descriptives
    return df.describe()

explore_describe(df)

,id,Date,Beat,district,Ward,community_area,x_coordinate,y_coordinate,Year,Latitude,Longitude,Month,Hour,YearMonth
count,1.000000e+04,10000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,1.000000e+04,10000.0,10000.000000,10000.000000,10000.0,10000.0,10000
mean,1.419741e+07,2026-05-31 09:56:41.856000,1142.839800,11.196600,22.722100,36.163800,1.165415e+06,1.887408e+06,2026.0,41.846628,-87.668464,5.4585,12.435,2026-05-15 05:07:26.400000
min,2.916800e+04,2026-05-23 20:30:00,111.000000,1.000000,1.000000,1.000000,1.100317e+06,1.814512e+06,2026.0,41.645796,-87.906463,5.0,0.0,2026-05-01 00:00:00
25%,1.421157e+07,2026-05-27 13:02:15,533.000000,5.000000,10.000000,23.000000,1.153952e+06,1.860744e+06,2026.0,41.773165,-87.710106,5.0,7.0,2026-05-01 00:00:00
50%,1.421581e+07,2026-05-31 10:30:00,1111.000000,11.000000,23.000000,31.000000,1.167098e+06,1.894432e+06,2026.0,41.866128,-87.662454,5.0,13.0,2026-05-01 00:00:00
75%,1.422004e+07,2026-06-04 08:01:15,1712.250000,17.000000,34.000000,53.000000,1.176890e+06,1.909212e+06,2026.0,41.906643,-87.626343,6.0,18.0,2026-06-01 00:00:00
max,1.422818e+07,2026-06-08 00:00:00,2535.000000,25.000000,50.000000,77.000000,1.205116e+06,1.951492e+06,2026.0,42.022527,-87.524533,6.0,23.0,2026-06-01 00:00:00
std,5.112256e+05,NaN,694.769575,6.942245,13.774805,21.120151,1.621136e+04,3.065574e+04,0.0,0.084308,0.059030,0.4983,7.057319,NaN


##  Requête 1a — Top 10 des types de crimes

On groupe les incidents par `Primary Type` et on compte le nombre d'occurrences pour identifier les crimes les plus fréquents à Chicago.

In [9]:
# Fonction : plot_top10_crimes
# Input  : df = DataFrame des crimes
# Output : figure Plotly (barres horizontales) du Top 10 des types de crimes
def plot_top10_crimes(df):
    top10 = (
        df.groupby("Primary Type")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(10)
    )

    fig = px.bar(
        top10,
        x="count",
        y="Primary Type",
        orientation="h",
        title="Top 10 des types de crimes à Chicago",
        color="count",
        color_continuous_scale="reds",
        labels={"count": "Nombre d'incidents", "Primary Type": "Type de crime"}
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    return fig

fig_top10 = plot_top10_crimes(df)
fig_top10.show()

##  Requête 1b — Arrestations par ward

On filtre les incidents ayant donné lieu à une arrestation (`Arrest == True`) et on les agrège par ward pour identifier les circonscriptions les plus actives.

In [10]:
# Fonction : plot_arrest_rate
# Input  : df = DataFrame des crimes
# Output : figure Plotly (barres horizontales) du taux d'arrestation par type de crime
def plot_arrest_rate(df):
    arrest_rate = (
        df.groupby("Primary Type")
        .agg(total=("Arrest", "count"), arrested=("Arrest", "sum"))
        .reset_index()
    )
    arrest_rate["taux"] = (arrest_rate["arrested"] / arrest_rate["total"] * 100).round(1)
    arrest_rate = arrest_rate.sort_values("taux", ascending=False)

    fig2 = px.bar(
        arrest_rate,
        x="taux",
        y="Primary Type",
        orientation="h",
        title="Taux d'arrestation par type de crime (%)",
        color="taux",
        color_continuous_scale="blues",
        labels={"taux": "Taux d'arrestation (%)", "Primary Type": "Type de crime"}
    )
    fig2.update_layout(yaxis={"categoryorder": "total ascending"})
    return fig2

fig_arrest = plot_arrest_rate(df)
fig_arrest.show()


##  Synthèse de l'exploration

- Le dataset couvre plusieurs années de données criminelles à Chicago
- Les colonnes `Latitude` et `Longitude` peuvent contenir des valeurs manquantes à prendre en compte pour l'analyse spatiale (Chrisa)
- La colonne `Date` est bien parsée en datetime → prête pour le forecasting (Léora)
- Les types de crimes les plus fréquents et la répartition par ward sont identifiés

---

## 6. Requête 2 — Pattern Mining & Analyse Avancée
**Auteure : Ekta**

**Objectif :** Identifier des associations récurrentes entre les caractéristiques 
des crimes (type, moment de la journée, lieu, arrestation) grâce à l'algorithme Apriori.


## Étape 1 : Discrétisation des variables

Avant d'appliquer l'algorithme Apriori, on doit transformer les variables continues 
ou textuelles en **catégories discrètes** :

| Variable | Transformation |
|---|---|
| `Date` (heure) | matin (6h-12h) / après-midi (12h-18h) / soir (18h-23h) / nuit (0h-6h) |
| `Primary Type` | gardé tel quel (type de crime) |
| `Location Description` | simplifié en grandes catégories (rue, intérieur, transport...) |
| `Arrest` | Arrestation_OUI / Arrestation_NON |
| `Domestic` | Domestique_OUI / Domestique_NON |

Cette étape est indispensable car Apriori travaille uniquement sur des données **booléennes**.

In [11]:
#fonction de discrétisation des variables
# Input : DataFrame brut
# Output : DataFrame avec colonnes discrétisées

def discretiser(df):
    df = df.copy()
    
    # 1. Discrétisation de l'heure en 4 tranches
    heure = df["Date"].dt.hour
    def tranche_horaire(h):
        if 6 <= h < 12:
            return "Matin"
        elif 12 <= h < 18:
            return "Après-midi"
        elif 18 <= h < 23:
            return "Soir"
        else:
            return "Nuit"
    df["Heure"] = heure.apply(tranche_horaire)
    
    # 2. Simplification du lieu (garder les 5 lieux les plus fréquents, regrouper le reste)
    top_lieux = df["Location Description"].value_counts().nlargest(5).index
    df["Lieu"] = df["Location Description"].apply(
        lambda x: x if x in top_lieux else "AUTRE"
    )
    
    # 3. Arrestation en texte lisible
    df["Arrestation"] = df["Arrest"].apply(lambda x: "Arrestation_OUI" if x else "Arrestation_NON")
    
    # 4. Crime domestique
    df["Domestique"] = df["Domestic"].apply(lambda x: "Domestique_OUI" if x else "Domestique_NON")

    df_disc = df[["Primary Type", "Heure", "Lieu", "Arrestation", "Domestique"]]
    
    return df_disc

df_disc = discretiser(df)
print("Discrétisation terminée")
print(df_disc["Heure"].value_counts())
print(df_disc["Lieu"].value_counts())
df_disc.head()

Discrétisation terminée
Heure
Après-midi    2841
Nuit          2646
Soir          2538
Matin         1975
Name: count, dtype: int64
Lieu
STREET                3152
AUTRE                 3076
APARTMENT             1827
RESIDENCE             1064
SIDEWALK               529
SMALL RETAIL STORE     352
Name: count, dtype: int64


,Primary Type,Heure,Lieu,Arrestation,Domestique
0,BATTERY,Nuit,APARTMENT,Arrestation_NON,Domestique_OUI
1,BATTERY,Nuit,RESIDENCE,Arrestation_NON,Domestique_OUI
2,MOTOR VEHICLE THEFT,Nuit,STREET,Arrestation_NON,Domestique_NON
3,MOTOR VEHICLE THEFT,Nuit,STREET,Arrestation_NON,Domestique_NON
4,OTHER OFFENSE,Nuit,RESIDENCE,Arrestation_NON,Domestique_NON


### Résultats de la discrétisation

**Répartition temporelle :**
- L'après-midi (12h-18h) est la tranche horaire la plus criminogène avec 2841 incidents (28%)
- La nuit (0h-6h) arrive en 2ème position avec 2646 incidents  car ces crimes sont souvent plus difficiles à résoudre
- Le matin reste la période la plus calme avec 1975 incidents

**Lieux les plus fréquents :**
- La **rue (STREET)** est le lieu le plus dangereux avec 3152 incidents
- Les **appartements (APARTMENT)** arrivent en 3ème position avec 1827 incidents,
  ce qui suggère une forte présence de crimes domestiques
- Les petits commerces (SMALL RETAIL STORE) ferment la liste avec 352 incidents

In [12]:
# Fonction d'encodage en format binaire (one-hot)
# Input : DataFrame discrétisé
# Output : DataFrame binaire prêt pour Apriori

def encoder_transactions(df_disc):
    #transforme chaque ligne en liste d'items
    transactions = df_disc.apply(lambda row: list(row.values), axis=1).tolist()
    
    te = TransactionEncoder()
    te_array = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_array, columns=te.columns_)
    
    return df_encoded

df_encoded = encoder_transactions(df_disc)
print(f"Matrice encodée : {df_encoded.shape[0]} lignes × {df_encoded.shape[1]} colonnes")
df_encoded.head()

Matrice encodée : 10000 lignes × 42 colonnes


,APARTMENT,ARSON,ASSAULT,AUTRE,Après-midi,Arrestation_NON,Arrestation_OUI,BATTERY,BURGLARY,CONCEALED CARRY LICENSE VIOLATION,CRIMINAL DAMAGE,CRIMINAL SEXUAL ASSAULT,CRIMINAL TRESPASS,DECEPTIVE PRACTICE,Domestique_NON,Domestique_OUI,GAMBLING,HOMICIDE,INTERFERENCE WITH PUBLIC OFFICER,INTIMIDATION,KIDNAPPING,LIQUOR LAW VIOLATION,MOTOR VEHICLE THEFT,Matin,NARCOTICS,Nuit,OBSCENITY,OFFENSE INVOLVING CHILDREN,OTHER OFFENSE,PROSTITUTION,PUBLIC INDECENCY,PUBLIC PEACE VIOLATION,RESIDENCE,ROBBERY,SEX OFFENSE,SIDEWALK,SMALL RETAIL STORE,STALKING,STREET,Soir,THEFT,WEAPONS VIOLATION
0,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
3,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
4,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False


## Étape 2 : Algorithme Apriori & Règles d'association



L'algorithme Apriori cherche des combinaisons d'items qui apparaissent souvent 
ensemble dans le dataset. On l'applique ici pour extraire des itemsets fréquents, 
puis générer des règles d'association mesurées par confidence et lift.

Nous choisissons **σ = 0.05** comme support minimal, ce qui signifie qu'une association 
doit apparaître dans au moins 5% des transactions, soit environ 500 incidents sur 10 000.
Ce seuil est un bon compromis : suffisamment bas pour capturer des patterns variés 
(119 itemsets, 324 règles), mais assez élevé pour éliminer les coïncidences rares 
non significatives.

In [13]:
# Fonction d'extraction des itemsets fréquents et règles d'association
# Input : DataFrame encodé, support minimal
# Output : DataFrame des règles d'association

def appliquer_apriori(df_encoded, min_support=0.05):
    # Extraction des itemsets fréquents
    itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
    itemsets = itemsets.sort_values("support", ascending=False)
    
    # Génération des règles d'association
    regles = association_rules(itemsets, metric="lift", min_threshold=1.0)
    regles = regles.sort_values("lift", ascending=False)
    
    print(f"Itemsets fréquents trouvés : {len(itemsets)}")
    print(f"Règles d'association trouvées : {len(regles)}")
    
    return itemsets, regles

itemsets, regles = appliquer_apriori(df_encoded, min_support=0.05)
regles[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

Itemsets fréquents trouvés : 119
Règles d'association trouvées : 324


,antecedents,consequents,support,confidence,lift
317,(Domestique_OUI),"(APARTMENT, BATTERY)",0.0512,0.262161,4.311855
316,"(APARTMENT, BATTERY)",(Domestique_OUI),0.0512,0.842105,4.311855
272,(MOTOR VEHICLE THEFT),"(Domestique_NON, STREET, Arrestation_NON)",0.0589,0.721814,2.909366
265,"(Domestique_NON, STREET, Arrestation_NON)",(MOTOR VEHICLE THEFT),0.0589,0.237404,2.909366
319,(BATTERY),"(Domestique_OUI, APARTMENT)",0.0512,0.251597,2.764803
314,"(Domestique_OUI, APARTMENT)",(BATTERY),0.0512,0.562637,2.764803
266,"(MOTOR VEHICLE THEFT, Domestique_NON)","(STREET, Arrestation_NON)",0.0589,0.734414,2.718038
271,"(STREET, Arrestation_NON)","(MOTOR VEHICLE THEFT, Domestique_NON)",0.0589,0.217987,2.718038
52,(Domestique_OUI),(BATTERY),0.1075,0.550435,2.704841
53,(BATTERY),(Domestique_OUI),0.1075,0.528256,2.704841


### Résultats de l'algorithme Apriori (σ=0.05)

L'algorithme a extrait **119 itemsets fréquents** et généré **324 règles d'association**.

### Top règles par lift — Interprétation

**Règle 1 — La plus forte (lift = 4.31) :**
> "Si BATTERY dans un APARTMENT → crime domestique"
- Confidence = 0.84 : dans 84% des cas, une agression en appartement est un crime domestique
- Lift = 4.31 : cette association est 4x plus fréquente que par hasard → très significatif
- Interprétation : les violences physiques en appartement sont quasi-systématiquement 
  des affaires familiales ou conjugales

**Règle 2 (lift = 2.91) :**
> "Si MOTOR VEHICLE THEFT → crime non domestique, sans arrestation, dans la rue"
- Confidence = 0.72 : dans 72% des cas, un vol de voiture a lieu dans la rue sans arrestation
- Lift = 2.91 : association 3x plus forte que le hasard
- Interprétation : les vols de véhicules sont des crimes opportunistes commis dans 
  l'espace public et rarement résolus par une arrestation immédiate

**Règle 3 (lift = 2.70) :**
> "Si BATTERY → crime domestique"
- Support = 0.107 : concerne plus de 10% de tous les incidents
- Interprétation : les agressions physiques so!nt fortement liées au cadre familial 
  à Chicago, ce qui pointe vers un problème structurel de violences conjugales/familiales

### Conclusion générale
Le pattern mining révèle deux profils criminels distincts :
- Les crimes **en intérieur (appartement)** → violence domestique, difficile à détecter
- Les crimes **en extérieur (rue)** → vols de véhicules, rarement élucidés sur le moment

## Impact du support minimal sur le nombre de patterns

On teste différentes valeurs de support minimal σ et on trace le nombre d'itemsets fréquents trouvés pour chaque valeur.
Ceci nous permet de choisir un bon compromis entre quantité et pertinence des règles.

In [14]:
# Fonction pour analyser l'impact du support minimal sur le nombre de patterns
# Input : DataFrame encodé
# Output : graphique Plotly

def graphique_support(df_encoded):
    supports = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
    nb_itemsets = []
    
    for s in supports:
        items = apriori(df_encoded, min_support=s, use_colnames=True)
        nb_itemsets.append(len(items))
    
    fig = px.line(
        x=supports,
        y=nb_itemsets,
        markers=True,
        title="Impact du support minimal σ sur le nombre d'itemsets fréquents",
        labels={"x": "Support minimal σ", "y": "Nombre d'itemsets fréquents"}
    )
    fig.add_vline(
        x=0.05, 
        line_dash="dash", 
        line_color="red",
        annotation_text="σ choisi = 0.05"
    )
    fig.show()
    return fig

fig_support = graphique_support(df_encoded)

Le graphique montre une chute brutale entre σ=0.05 (119 itemsets) et σ=0.1 (44 itemsets).
Au-delà de σ=0.1, la courbe s'aplatit progressivement jusqu'à seulement 3 itemsets à σ=0.4.

Nous avons choisi σ=0.05 car c'est le meilleur compromis :
- Assez bas pour capturer des associations intéressantes (119 itemsets, 324 règles)
- Assez élevé pour éliminer les coïncidences rares non significatives
- Correspond à des items présents dans au moins ~500 transactions sur 10 000

## Étape 3 — Visualisation Sankey

Le **diagramme de Sankey** permet de visualiser les règles d'association :
Dans ce graph les antécédents sont à gauche, les conséquents à droite.
L'épaisseur des liens représente la confidence, la couleur représente le lift.
On garde les top_n règles triées par lift pour ne garder que les plus intéressantes.

In [15]:
# Fonction de visualisation Sankey des règles d'association
# Input : DataFrame des règles
# Output : graphique Sankey Plotly

def sankey_regles(regles, top_n=15):
    """
    Visualise les top_n règles d'association sous forme de diagramme Sankey.
    Les antécédents sont à gauche, les conséquents à droite.
    On filtre les règles réciproques pour éviter les boucles.
    """
    top_regles = regles.nlargest(top_n, "lift").copy()
    
    top_regles["ant_str"] = top_regles["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
    top_regles["cons_str"] = top_regles["consequents"].apply(lambda x: ", ".join(sorted(list(x))))
    
    # Filtrer les boucles (antécédent == conséquent)
    top_regles = top_regles[top_regles["ant_str"] != top_regles["cons_str"]]
    
    # Filtrer les règles réciproques (garder seulement la meilleure direction)
    seen = set()
    rows = []
    for _, row in top_regles.iterrows():
        pair = frozenset([row["ant_str"], row["cons_str"]])
        if pair not in seen:
            seen.add(pair)
            rows.append(row)
    top_regles = pd.DataFrame(rows)
    
    all_labels = list(set(top_regles["ant_str"].tolist() + top_regles["cons_str"].tolist()))
    label_idx = {label: i for i, label in enumerate(all_labels)}
    
    sources = [label_idx[a] for a in top_regles["ant_str"]]
    targets = [label_idx[c] for c in top_regles["cons_str"]]
    values = top_regles["confidence"].tolist()
    lifts = top_regles["lift"].tolist()
    
    colors = [f"rgba(255, {max(0, int(255 - l*30))}, 0, 0.5)" for l in lifts]
    
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=30,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_labels,
            color="lightblue"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors
        )
    ))
    
    fig.update_layout(
        title_text="Règles d'association — Diagramme de Sankey (épaisseur = confidence, couleur = lift)",
        font_size=11,
        height=600,
        width=1000
    )
    fig.show()
    return fig

fig_sankey = sankey_regles(regles, top_n=15)

### Lecture du diagramme de Sankey

Chaque flux représente une règle d'association :
- **Gauche** = antécédent (condition SI)
- **Droite** = conséquent (conclusion ALORS)
- **Épaisseur du lien** = confidence (plus c'est épais, plus la règle est fiable)
- **Couleur** = lift (orange foncé = association très forte, jaune = association modérée)

**Ce qu'on observe :**
- **MOTOR VEHICLE THEFT** → mène systématiquement vers une absence d'arrestation 
  dans la rue (lien très épais = confidence élevée)
- **BATTERY + Domestique_OUI** → fortement associé aux appartements, 
  confirmant le profil de violence conjugale/familiale
- **APARTMENT, BATTERY** → transite par **Domestique_OUI** avant de mener 
  vers d'autres caractéristiques, montrant que ce nœud est central dans 
  les crimes domestiques

## Conclusion Pattern Mining

Cette analyse par règles d'association a permis d'identifier des patterns récurrents 
dans la criminalité de Chicago sur 10 000 incidents récents.

Les principaux enseignements sont :
- Les **agressions (BATTERY) en appartement** sont quasi-systématiquement des crimes 
  domestiques (lift = 4.31): ce qui pointe vers un problème structurel de violences familiales
- Les **vols de véhicules (MOTOR VEHICLE THEFT)** ont lieu dans la rue sans arrestation 
  dans 72% des cas , des crimes opportunistes difficiles à résoudre immédiatement
- Les crimes commis **l'après-midi** sont les plus fréquents avec 2841 incidents

Ces patterns constituent une base solide pour anticiper les zones et situations à risque, 
et peuvent aider à mieux orienter les ressources policières sur le terrain.

**Prompt LLM utilisé :** Ce notebook a été développé avec l'assistance de Claude (Anthropic) 
pour la structure du code, les visualisations et les interprétations des métriques Apriori.

---

## 7. Requête 3 — Analyse temporelle & Forecasting
**Auteure : Léora**

**Objectif :** Analyser l'évolution temporelle des crimes à Chicago et prévoir 
leur tendance future grâce au modèle de forecasting Prophet.



In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet

### Chargement du dataset

In [17]:
# Fonction de chargement du dataset
# Input : chemin vers le fichier CSV
# Output : dataframe pandas avec la colonne Date parsée en datetime
def load_data_bridgeton(file_path):
    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"], format='mixed')
    print(f"Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
    return df

In [22]:
filepath="../data/chicago_crime.csv"
df_temp = load_data_bridgeton(filepath)
df_temp.head()

Dataset chargé : 949 lignes × 20 colonnes


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,Ward,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14216537,JK278149,2026-06-01 21:58:00,007XX W 31ST ST,0560,ASSAULT,SIMPLE,DRUG STORE,True,False,915,11,08A,1171697.0,1884328.0,2026,2026 Jun 09 04:13:19 PM,"41,83805499","-87,645458414",POINT (-87.645458414 41.83805499)
1,14203870,JK262663,2026-05-20 17:50:00,007XX W 31ST ST,0850,THEFT,ATTEMPT THEFT,SIDEWALK,False,False,915,11,06,1171668.0,1884327.0,2026,2026 May 28 03:42:02 PM,"41,838052883","-87,645564858",POINT (-87.645564858 41.838052883)
2,14178251,JK231486,2026-04-25 22:55:00,007XX W 31ST ST,0860,THEFT,RETAIL THEFT,CONVENIENCE STORE,False,False,915,11,06,1171697.0,1884328.0,2026,2026 May 03 03:43:04 PM,"41,83805499","-87,645458414",POINT (-87.645458414 41.83805499)
3,14134795,JK178608,2026-03-12 15:07:00,007XX W 31ST ST,0560,ASSAULT,SIMPLE,STREET,False,False,915,11,08A,1171697.0,1884328.0,2026,2026 Mar 20 03:43:34 PM,"41,83805499","-87,645458414",POINT (-87.645458414 41.83805499)
4,14114764,JK154429,2026-02-18 18:30:00,007XX W 31ST ST,0820,THEFT,$500 AND UNDER,SMALL RETAIL STORE,False,False,915,11,06,1171668.0,1884327.0,2026,2026 Mar 14 03:41:39 PM,"41,838052883","-87,645564858",POINT (-87.645564858 41.838052883)


## Analyse temporelle

### Évolution mensuelle des crimes

Ce graphique représente le nombre de crimes commis à Bridgeport (Chicago) 
répartis par mois sur toute la période disponible (2002-2026).

**Calcul :** Les données sont agrégées par mois grâce à la fonction 
`monthly_aggregation()` qui groupe les incidents par année et mois.

**Interprétation :** On observe un pic en juin 2005 avec 11 crimes. 
Après 2006, le nombre de crimes tend à diminuer globalement. 
La criminalité reste variable d'un mois à l'autre mais stable sur le long terme,
avec une légère saisonnalité visible.

In [ ]:
# Fonction d'agrégation des crimes par mois
# input : Dataframe pandas avec colonne Date en datetime
# Output : dataframe avec le nombre de crimes par mois

def monthly_aggregation(df_temp):
    df_temp["YearMonth"] = df_temp["Date"].dt.to_period("M")
    monthly = df_temp.groupby("YearMonth").size().reset_index(name="Crime_Count")
    monthly["YearMonth"] = monthly["YearMonth"].dt.to_timestamp()
    return monthly

monthly = monthly_aggregation(df_temp)
monthly.head()

In [ ]:
# Fonction de visualisation de l'évolution mensuelle des crimes
# Input : dataframe mensuel avec colonnes YearMonth et Crime_Count
# Output : figure Plotly

def plot_monthly_crimes(monthly):
    # Trouver le mois avec le plus de crimes
    idx_max = monthly["Crime_Count"].idxmax()
    pic = monthly.loc[idx_max]
    
    plot = px.line(
        monthly,
        x="YearMonth",
        y="Crime_Count",
        title="Evolution mensuelle des crimes à Chicago",
        labels={"YearMonth": "Année", "Crime_Count": "Nombre de crimes"}
    )
    
    # Annoter le pic maximum sur le graphique
    plot.add_annotation(
        x=pic["YearMonth"],
        y=pic["Crime_Count"],
        text=f"Pic : {pic['Crime_Count']} crimes ({pic['YearMonth'].strftime('%b %Y')})",
        showarrow=True,
        arrowhead=2,
        yshift=10
    )
    return plot

fig_monthly = plot_monthly_crimes(monthly)
fig_monthly.show()

### Évolution annuelle des crimes

Ce graphique représente le nombre total de crimes commis à Bridgeport (Chicago) 
par année sur la période 2002-2026.

**Calcul :** Les données sont agrégées par année grâce à la fonction 
`yearly_aggregation()` qui groupe les incidents par année et compte le nombre 
d'occurrences.

**Interprétation :** On observe un pic record en 2011 avec 66 crimes. 
Après cette année, la criminalité suit une tendance globalement baissière 
jusqu'en 2022. La chute observée en 2026 s'explique par le fait que l'année 
n'est pas encore terminée au moment de l'analyse.

In [ ]:
# Fonction d'agrégation des crimes par année
# input : Dataframe pandas avec colonne Date en datetime
# Output : dataframe avec le nombre de crimes par année
def yearly_aggregation(df_temp):
    df_temp["Year"] = df_temp["Date"].dt.year
    yearly = df_temp.groupby("Year").size().reset_index(name="Crime_Count")
    return yearly

yearly = yearly_aggregation(df_temp)
yearly.head()

In [ ]:
# Fonction de visualisation de l'évolution annuelle des crimes
# Input : dataframe annuel avec colonnes Year et Crime_Count
# Output : figure Plotly

def plot_yearly_crimes(yearly):
    # Trouver l'année avec le plus de crimes
    idx_max = yearly["Crime_Count"].idxmax()
    pic = yearly.loc[idx_max]
    
    plot = px.line(
        yearly,
        x="Year",
        y="Crime_Count",
        title="Evolution annuelle des crimes à Chicago",
        labels={"Year": "Année", "Crime_Count": "Nombre de crimes"}
    )
    
    # Annoter le pic maximum sur le graphique
    plot.add_annotation(
        x=pic["Year"],
        y=pic["Crime_Count"],
        text=f"Pic : {pic['Crime_Count']} crimes ({pic['Year']})",
        showarrow=True,
        arrowhead=2,
        yshift=10
    )
    return plot

fig_yearly = plot_yearly_crimes(yearly)
fig_yearly.show()

### Prévision du nombre de crimes avec Prophet

Ce graphique représente l'évolution passée et la prévision future du nombre 
de crimes mensuels à Bridgeport (Chicago) sur les 12 prochains mois.

**Calcul :** Le modèle Prophet (développé par Facebook) est entraîné sur 
les données historiques mensuelles. Il prédit automatiquement la tendance 
et la saisonnalité à partir des données passées.

**Indicateurs :**
- `yhat` : valeur prédite (nombre de crimes attendu)
- `yhat_lower` : borne basse de l'intervalle de confiance (scénario optimiste)
- `yhat_upper` : borne haute de l'intervalle de confiance (scénario pessimiste)

**Interprétation :** Le modèle prédit que la criminalité restera stable, 
avec une légère tendance à la baisse. Le nombre de crimes continuera de 
varier selon les mois (saisonnalité) mais restera globalement stable d'une 
année à l'autre.

In [ ]:
df_prophet = monthly.rename(columns={"YearMonth": "ds", "Crime_Count": "y"})

In [ ]:
# Fonction de forecasting avec Prophet
# Input : dataframe avec colonnes ds (dates) et y (nombre de crimes)
# Output : dataframe de prévisions avec yhat, yhat_lower, yhat_upper
def run_prophet(df_prophet):
    model = Prophet()
    model.fit(df_prophet)
    future = model.make_future_dataframe(periods=12, freq="MS")
    forecast = model.predict(future)
    return model, forecast
model, forecast = run_prophet(df_prophet)

In [ ]:
# Fonction de forecasting avec Prophet
# Input : forecast prophet et dataframe adapté
# Output : graphique plotly
def plot_forecast(forecast, df_prophet):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=df_prophet["ds"], y=df_prophet["y"], name="Données réelles", mode="lines"))
    fig.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat"], name="Prédiction", mode="lines"))
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast["ds"], forecast["ds"][::-1]]),
        y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"][::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.1)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Intervalle de confiance"
    ))

    fig.update_layout(
        title="Prévision du nombre de crimes à Chicago",
        xaxis_title="Date",
        yaxis_title="Nombre de crimes"
    )
    return fig

plot_forecast(forecast, df_prophet).show()

### Composantes du modèle Prophet

Ces graphiques décomposent la prévision en deux éléments distincts.

**Tendance :** La criminalité suit une tendance globalement descendante 
depuis 2011, confirmant la baisse observée sur le graphique annuel.

**Saisonnalité annuelle :** On observe une variation cyclique au cours 
de l'année avec un pic en juillet (été) et un creux en janvier/février (hiver). 
Cela suggère que la criminalité est influencée par les saisons.

In [ ]:
# Fonction de visualisation des composantes Prophet (tendance + saisonnalité)
# Input : modèle Prophet entraîné et dataframe de prévisions
# Output : figure matplotlib
def plot_prophet_components(model, forecast):
    fig = model.plot_components(forecast)
    return fig

plot_prophet_components(model, forecast)

---

## 8. Requête 4 — Analyse spatiale & Clustering
**Auteure : Chrisa**

**Objectif :** Identifier les zones géographiques de Chicago où la criminalité se concentre,
grâce à la visualisation spatiale et aux algorithmes de clustering K-means et DBSCAN.



In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import DBSCAN, KMeans
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

### Préparation des données spatiales

On filtre les lignes sans coordonnées GPS et hors de la zone géographique de Chicago.
On convertit ensuite chaque paire (longitude, latitude) en objet `Point` géoréférencé.

**ATTENTION :** convention `Point(longitude, latitude)` — longitude en premier.
La projection `EPSG:4326` = WGS84, le standard GPS mondial.

In [ ]:
# Fonction : prepare_spatial_data
# Input  : DataFrame complet
# Output : DataFrame avec uniquement les observations géolocalisables dans Chicago
def prepare_spatial_data(df):
    spatial = df.dropna(subset=["Latitude", "Longitude"]).copy()
    # Filtre géographique : Chicago est entre lat 41.6-42.1 et lon -87.9/-87.5
    return spatial[
        spatial["Latitude"].between(41.6, 42.1) &
        spatial["Longitude"].between(-87.9, -87.5)
    ].copy()


# Fonction : create_geodataframe
# Input  : DataFrame spatial avec colonnes Latitude et Longitude
# Output : GeoDataFrame avec objets Point géoréférencés (EPSG:4326)
def create_geodataframe(spatial_df):
    # Point(longitude, latitude) — longitude en premier (convention géographique)
    geometry = [Point(lon, lat) for lon, lat in
                zip(spatial_df["Longitude"], spatial_df["Latitude"])]
    gdf = gpd.GeoDataFrame(spatial_df.copy(), geometry=geometry, crs="EPSG:4326")
    print(f"GeoDataFrame créé : {len(gdf)} points | Projection : {gdf.crs}")
    return gdf


spatial_df = prepare_spatial_data(df)
gdf        = create_geodataframe(spatial_df)
gdf[["Primary Type", "Arrest", "Latitude", "Longitude", "geometry"]].head()

## Carte de densité des crimes

Ce graphique représente la concentration géographique des crimes à Chicago.
En zoomant, on voit des zones quasi noires correspondant aux hotspots maximaux,
notamment dans le West Side et le long de la côte est.

**Calcul :** Pour chaque point de la carte, Plotly compte le nombre de crimes
dans un rayon de `radius=10` pixels et colore la zone en conséquence.

**Interprétation :** Plus la couleur est foncée, plus la densité de crimes est élevée.
La criminalité n'est pas uniformément répartie — elle forme des clusters géographiques précis.

In [ ]:
# Fonction : plot_density_map
# Input  : DataFrame spatial avec colonnes Latitude et Longitude
# Output : carte de densité interactive Plotly
def plot_density_map(spatial_df):
    fig = px.density_mapbox(
        spatial_df,
        lat="Latitude",
        lon="Longitude",
        radius=10,
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="Densité spatiale des crimes à Chicago",
        color_continuous_scale="Reds"
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig

plot_density_map(spatial_df).show()

## Clustering K-means

K-means divise les crimes en `k=6` zones géographiques en regroupant les points les plus proches.
On choisit k=6 car Chicago est naturellement divisée en 6 grands secteurs
(North, Northwest, West, Central, South, Far South).

**Calcul :** L'algorithme place 6 centres, assigne chaque crime au centre le plus proche,
déplace les centres vers le milieu de leur groupe, et répète jusqu'à convergence.

**Interprétation :** Chaque couleur représente une zone géographique.
Les zones 0 et 2 concentrent le plus de crimes (plus de 2000 incidents chacune).
La zone 1 est la moins touchée (environ 1000 incidents).

**Limite :** K-means force tous les crimes dans une zone, même les crimes isolés.

In [ ]:
# Fonction : apply_kmeans
# Input  : DataFrame spatial, k = nombre de clusters
# Output : DataFrame avec colonne cluster_kmeans, modèle KMeans entraîné
def apply_kmeans(spatial_df, k=6):
    coords = spatial_df[["Latitude", "Longitude"]].values
    model  = KMeans(n_clusters=k, random_state=42, n_init=10)
    df     = spatial_df.copy()
    # .astype(str) pour que Plotly affiche des couleurs distinctes (et non un dégradé)
    df["cluster_kmeans"] = model.fit_predict(coords).astype(str)
    print(f"K-means : {k} zones créées")
    print(df["cluster_kmeans"].value_counts().sort_index())
    return df, model


# Fonction : plot_kmeans_map
# Input  : DataFrame avec colonne cluster_kmeans
# Output : carte interactive colorée par zone K-means
def plot_kmeans_map(kmeans_df):
    fig = px.scatter_mapbox(
        kmeans_df,
        lat="Latitude",
        lon="Longitude",
        color="cluster_kmeans",
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="K-means — 6 zones de criminalité à Chicago",
        hover_data=["Primary Type", "Arrest"],
        opacity=0.6
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig


df_kmeans, kmeans_model = apply_kmeans(spatial_df, k=6)
plot_kmeans_map(df_kmeans).show()

## Clustering DBSCAN

Contrairement à K-means, DBSCAN détecte automatiquement les zones denses
sans qu'on fixe le nombre de clusters à l'avance.
Les crimes isolés sont étiquetés **-1** et apparaissent en gris sur la carte.

On a d'abord essayé OPTICS mais avec 10 000 points GPS sur Chicago,
l'algorithme mettait tout dans un seul cluster car il considérait que tout était dense.
DBSCAN est plus adapté car il permet de contrôler le rayon de recherche en kilomètres.

**Paramètres :**
- `eps=0.5/6371` : rayon de 0.5 km converti en radians (6371 = rayon de la Terre en km)
- `min_samples=50` : minimum 50 crimes dans ce rayon pour former une zone dense
- `metric="haversine"` : calcule les vraies distances GPS sur la sphère terrestre

**Interprétation :** Les hotspots colorés se concentrent dans le West Side et le long de
la côte est — confirmant exactement ce qu'on voyait sur la heatmap.
Les points gris (-1) sont des crimes géographiquement isolés, répartis sur toute la ville.

In [ ]:
# Fonction : apply_dbscan
# Input  : DataFrame spatial, eps = rayon en km, min_samples = densité minimale
# Output : DataFrame avec colonne cluster_dbscan (-1 = crime isolé), modèle DBSCAN
def apply_dbscan(spatial_df, eps=0.5, min_samples=50):

    # Conversion en radians car haversine travaille en radians
    # 6371 = rayon de la Terre en km
    coords = np.radians(spatial_df[["Latitude", "Longitude"]].to_numpy())

    # eps en radians (0.5 km / 6371 km)
    # metric="haversine" calcule les vraies distances GPS sur la sphère terrestre
    model  = DBSCAN(eps=eps/6371, min_samples=min_samples,
                    metric="haversine", algorithm="ball_tree")
    result = spatial_df.copy()

    # .astype(str) pour que Plotly affiche des couleurs distinctes
    result["cluster_dbscan"] = model.fit_predict(coords).astype(str)

    nb_zones = result["cluster_dbscan"].nunique() - 1
    nb_bruit = (result["cluster_dbscan"] == "-1").sum()
    print(f"DBSCAN : {nb_zones} zones denses | {nb_bruit} crimes isolés ({nb_bruit/len(result)*100:.1f}%)")
    return result, model


# Fonction : plot_dbscan_map
# Input  : DataFrame avec colonne cluster_dbscan
# Output : carte interactive — hotspots colorés, crimes isolés en gris
def plot_dbscan_map(dbscan_df):

    # Séparer le bruit des clusters pour mieux visualiser
    bruit    = dbscan_df[dbscan_df["cluster_dbscan"] == "-1"]
    clusters = dbscan_df[dbscan_df["cluster_dbscan"] != "-1"]

    fig = go.Figure()

    # D'abord les crimes isolés en gris transparent en arrière-plan
    fig.add_trace(go.Scattermapbox(
        lat=bruit["Latitude"], lon=bruit["Longitude"],
        mode="markers",
        marker=dict(size=4, color="lightgrey", opacity=0.3),
        name="-1 (crimes isolés)"
    ))

    # Ensuite les clusters colorés par-dessus
    for cluster_id in sorted(clusters["cluster_dbscan"].unique()):
        subset = clusters[clusters["cluster_dbscan"] == cluster_id]
        fig.add_trace(go.Scattermapbox(
            lat=subset["Latitude"], lon=subset["Longitude"],
            mode="markers",
            marker=dict(size=6, opacity=0.8),
            name=f"Zone {cluster_id}"
        ))

    fig.update_layout(
        mapbox=dict(style="carto-positron",
                    center={"lat": 41.85, "lon": -87.65}, zoom=10),
        title="DBSCAN — Zones denses de criminalité à Chicago (gris = crime isolé)",
        height=600,
        margin={"r": 0, "t": 50, "l": 0, "b": 0}
    )
    return fig


df_dbscan, dbscan_model = apply_dbscan(spatial_df)
plot_dbscan_map(df_dbscan).show()

## Comparaison K-means vs DBSCAN

| Critère | K-means | DBSCAN |
|---|---|---|
| Nombre de zones | Fixé (k=6) | Automatique (18 trouvées) |
| Crimes isolés | Forcés dans une zone | Gris (-1) |
| Forme des zones | Ronde et régulière | Libre |
| Paramètre clé | k | eps (rayon en km) |

**Résultats observés :**
Les deux méthodes s'accordent sur les mêmes zones à risque — principalement le West Side
et la bande est de Chicago le long du lac Michigan.
K-means donne une vision globale en 6 grandes zones.
DBSCAN est plus précis et révèle 18 hotspots denses avec les crimes isolés en gris.

## Composition des types de crimes par zone

On enrichit le clustering K-means en regardant quels types de crimes dominent
dans chaque zone géographique.

**Interprétation :** Les clusters 0 et 2 sont les plus touchés (plus de 2000 incidents).
L'assault et la battery dominent dans toutes les zones, mais certaines ont
plus de motor vehicle theft ou de narcotics — chaque zone a un profil criminel distinct.

In [ ]:
# Fonction : plot_crime_type_by_cluster
# Input  : DataFrame avec colonne cluster_kmeans
# Output : graphique barres empilées — types de crimes par zone géographique
def plot_crime_type_by_cluster(kmeans_df):
    group = (
        kmeans_df.groupby(["cluster_kmeans", "Primary Type"])
        .size()
        .reset_index(name="count")
    )
    fig = px.bar(
        group,
        x="cluster_kmeans",
        y="count",
        color="Primary Type",
        title="Composition des types de crimes par zone (K-means)",
        labels={"cluster_kmeans": "Zone", "count": "Nombre de crimes",
                "Primary Type": "Type de crime"},
        barmode="stack"
    )
    return fig

plot_crime_type_by_cluster(df_kmeans).show()

## Synthèse

- La heatmap confirme que la criminalité se concentre sur la bande est de Chicago
  et dans le West Side — elle n'est pas uniformément répartie
- K-means identifie 6 zones géographiques avec des profils criminels distincts
- DBSCAN détecte automatiquement 18 hotspots précis et révèle les crimes géographiquement isolés
- Les deux méthodes s'accordent sur les mêmes zones à risque
- Cette analyse répond à la dimension **spatiale** de notre problématique

---
## 9. Intégration du dashboard

Le dashboard est un livrable distinct du notebook.
Il organise les résultats des quatre requêtes et doit être lancé depuis la racine du projet avec :

```bash
python dashboard/app.py
```

Le navigateur doit ensuite ouvrir : `http://127.0.0.1:8050`

In [19]:
# Fonction : launch_dashboard
# Input  : racine du projet, start_server = True pour démarrer Dash
# Output : chemin du fichier app.py ou processus serveur
def launch_dashboard(project_root: Path, start_server: bool = False):
    app_path = project_root / "dashboard" / "app.py"
    if not app_path.exists():
        print("Dashboard non trouvé. Ajoutez dashboard/app.py avant la remise finale.")
        return None
    if start_server:
        return subprocess.run([sys.executable, str(app_path)], check=True)
    print(f"Dashboard détecté : {app_path}")
    print("Commande de lancement : python dashboard/app.py")
    return app_path


PROJECT_ROOT     = resolve_project_root()
START_DASHBOARD  = False  # passer à True pendant la démo pour démarrer Dash
launch_dashboard(PROJECT_ROOT, start_server=START_DASHBOARD)

Dashboard détecté : D:\LeoraData\Documents\Ecole\ESILV\DataAnalysis\chicago-crime-analysis\dashboard\app.py
Commande de lancement : python dashboard/app.py


WindowsPath('D:/LeoraData/Documents/Ecole/ESILV/DataAnalysis/chicago-crime-analysis/dashboard/app.py')

---
## 10. Bloc principal consolidé & dashboard Dash

Le bloc principal ci-dessous appelle **toutes les fonctions** des quatre requêtes dans l'ordre,
puis lance le dashboard Dash interactif.

- Les Requêtes 1, 2 et 4 partagent le même chargement API (chargé une seule fois et réutilisé).
- La **Requête 3** utilise sa source CSV (`../data/chicago_crime.csv`) via `load_data_csv`.
- La fonction `launch_dashboard` regroupe les figures Plotly dans un dashboard multi-onglets.
  Le graphique des composantes Prophet est un objet Matplotlib : il n'est pas intégré au dashboard
  Dash (qui n'affiche que des figures Plotly) mais reste disponible dans le notebook.

In [ ]:
# Fonction : launch_dashboard
# Input  : les figures Plotly produites par les quatre requêtes
# Output : lance le serveur Dash (dashboard interactif multi-onglets)
def launch_dashboard(fig_top10, fig_arrest,
                     fig_support, fig_sankey,
                     fig_monthly, fig_yearly, fig_forecast,
                     fig_density, fig_kmeans, fig_dbscan, fig_crime_type):
    import dash
    from dash import dcc, html

    app = dash.Dash(__name__)
    app.title = "Chicago Crime Analysis"

    app.layout = html.Div([
        html.H1("Chicago Crime Analysis — Dashboard"),
        dcc.Tabs([
            dcc.Tab(label="Requête 1 — Exploration", children=[
                dcc.Graph(figure=fig_top10),
                dcc.Graph(figure=fig_arrest),
            ]),
            dcc.Tab(label="Requête 2 — Pattern Mining", children=[
                dcc.Graph(figure=fig_support),
                dcc.Graph(figure=fig_sankey),
            ]),
            dcc.Tab(label="Requête 3 — Analyse temporelle", children=[
                dcc.Graph(figure=fig_monthly),
                dcc.Graph(figure=fig_yearly),
                dcc.Graph(figure=fig_forecast),
            ]),
            dcc.Tab(label="Requête 4 — Analyse spatiale", children=[
                dcc.Graph(figure=fig_density),
                dcc.Graph(figure=fig_kmeans),
                dcc.Graph(figure=fig_dbscan),
                dcc.Graph(figure=fig_crime_type),
            ]),
        ])
    ])

    # Lancement du serveur (http://127.0.0.1:8050)
    app.run(debug=True, port=8050)

In [ ]:
# Bloc principal — appelle toutes les fonctions des 4 requêtes puis lance le dashboard Dash
if __name__ == "__main__":

    # ===================== Requête 1 — Exploration (source : API) =====================
    df_api = load_data()                       # API Chicago Data Portal
    explore_shape(df_api)
    explore_dtypes(df_api)
    explore_missing(df_api)
    explore_ranges(df_api)
    explore_describe(df_api)
    fig_top10  = plot_top10_crimes(df_api)
    fig_arrest = plot_arrest_rate(df_api)

    # ===================== Requête 2 — Pattern Mining (source : API) ==================
    df_disc    = discretiser(df_api)
    df_encoded = encoder_transactions(df_disc)
    itemsets, regles = appliquer_apriori(df_encoded, min_support=0.05)
    fig_support = graphique_support(df_encoded)
    fig_sankey  = sankey_regles(regles, top_n=15)

    # ===================== Requête 3 — Analyse temporelle (source : CSV) =============
    df_temp = load_data_bridgeton("../data/chicago_crime.csv")   # CSV (comme dans temporal_analysis.ipynb)
    monthly = monthly_aggregation(df_temp)
    fig_monthly = plot_monthly_crimes(monthly)
    yearly = yearly_aggregation(df_temp)
    fig_yearly = plot_yearly_crimes(yearly)
    df_prophet = monthly.rename(columns={"YearMonth": "ds", "Crime_Count": "y"})
    model, forecast = run_prophet(df_prophet)
    fig_forecast = plot_forecast(forecast, df_prophet)
    plot_prophet_components(model, forecast)   # composantes Prophet (Matplotlib)

    # ===================== Requête 4 — Analyse spatiale (source : API) ================
    spatial_df = prepare_spatial_data(df_api)
    gdf        = create_geodataframe(spatial_df)
    fig_density = plot_density_map(spatial_df)
    df_kmeans, kmeans_model = apply_kmeans(spatial_df, k=6)
    fig_kmeans  = plot_kmeans_map(df_kmeans)
    df_dbscan, dbscan_model = apply_dbscan(spatial_df)
    fig_dbscan  = plot_dbscan_map(df_dbscan)
    fig_crime_type = plot_crime_type_by_cluster(df_kmeans)

    print("✅ Toutes les analyses ont été exécutées.")

    # ===================== Lancement du dashboard Dash ===============================
    launch_dashboard(
        fig_top10, fig_arrest,
        fig_support, fig_sankey,
        fig_monthly, fig_yearly, fig_forecast,
        fig_density, fig_kmeans, fig_dbscan, fig_crime_type
    )